## 🎯 Learning Objectives
* Understand the fundamental components of a Retrieval-Augmented Generation (RAG) pipeline: Indexing, Retrieval, and Generation.
* Grasp the purpose and benefits of RAG in overcoming limitations of standalone Large Language Models (LLMs).
* Implement a basic RAG pipeline using modern Python libraries and tools.
* Analyze the flow of information and the role of each component in generating context-aware responses.


## The Core RAG Pipeline: Index, Retrieve, Generate

Welcome to RAG-01, where we demystify the powerful world of Retrieval-Augmented Generation (RAG). In this lesson, we'll dive into the foundational architecture that allows Large Language Models (LLMs) to move beyond their pre-trained knowledge and interact with up-to-date, domain-specific, and factual information. Think of RAG as giving your LLM a "super-powered research assistant" that can quickly find relevant information before the LLM formulates its answer.

### Why RAG?

Standalone LLMs, while incredibly powerful, have inherent limitations:

1.  **Knowledge Cut-off**: Their knowledge is limited to their training data, which quickly becomes outdated.
2.  **Hallucination**: They can generate plausible but incorrect or fabricated information.
3.  **Lack of Specificity**: They often lack deep, domain-specific knowledge required for specialized tasks.
4.  **Transparency**: It's hard to trace the source of their answers.

RAG addresses these by enabling LLMs to access and incorporate external, authoritative knowledge bases in real-time. This leads to more accurate, relevant, and verifiable responses.

### The Three Pillars of RAG: Index, Retrieve, Generate

The RAG pipeline is typically broken down into three core stages:

#### 1. Indexing (The Knowledge Preparation Phase)

Before an LLM can retrieve information, that information must be organized and stored efficiently. This phase is about transforming raw data into a searchable format.

*   **Document Loading**: Gathering your data from various sources (PDFs, databases, web pages, text files, etc.).
*   **Text Splitting (Chunking)**: Large documents are broken down into smaller, manageable chunks. This is crucial because embedding models have token limits, and smaller chunks allow for more precise retrieval.
*   **Embedding**: Each text chunk is converted into a numerical vector (an embedding) using an embedding model. These vectors capture the semantic meaning of the text. Chunks with similar meanings will have vectors that are numerically close to each other in a high-dimensional space.
*   **Vector Storage**: These embeddings, along with their original text chunks, are stored in a specialized database called a **Vector Database** (e.g., ChromaDB, Pinecone, Weaviate, Qdrant). Vector databases are optimized for fast similarity searches.

*Analogy*: Imagine you have a vast library (your data). Indexing is like hiring a team of librarians to read every book, summarize its key ideas into short notes (chunks), translate those notes into a universal classification code (embeddings), and then meticulously organize these codes and notes in a special catalog (vector database) that allows for lightning-fast lookups based on meaning, not just keywords.

#### 2. Retrieval (The Search Phase)

When a user asks a question, this phase is about finding the most relevant pieces of information from your indexed knowledge base.

*   **Query Embedding**: The user's query is also converted into an embedding using the *same* embedding model used during indexing. This ensures consistency in the vector space.
*   **Similarity Search**: The query embedding is then compared against all the document embeddings in the vector database. The system identifies the top 'k' most similar document chunks (based on cosine similarity or other distance metrics).
*   **Context Selection**: These retrieved chunks are the 


In [ ]:
# Install necessary libraries (run this cell once)
# !pip install -qU langchain sentence-transformers chromadb

import os
from langchain_community.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.embeddings import SentenceTransformerEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# --- 1. Indexing Phase: Prepare the Knowledge Base ---

print("--- Starting Indexing Phase ---")

# Sample Documents (representing your external knowledge base)
documents_content = [
    "The AgenticLabs.ng mission is to empower developers with cutting-edge AI and automation tools.",
    "RAG systems enhance LLMs by providing external, up-to-date information, reducing hallucinations.",
    "ChromaDB is a popular open-source vector database, ideal for local development and small-scale RAG applications.",
    "Large Language Models (LLMs) are powerful AI models capable of understanding and generating human-like text.",
    "The core RAG pipeline consists of three main steps: Index, Retrieve, and Generate.",
    "Agentic AI focuses on creating autonomous agents that can reason, plan, and act in complex environments.",
    "Vector embeddings convert text into numerical representations, capturing semantic meaning.",
    "Retrieval-Augmented Generation (RAG) combines the strengths of information retrieval with generative AI."
]

# For a real application, you'd load from files:
# with open("data/agenticlabs_mission.txt", "w") as f: f.write(documents_content[0])
# loader = TextLoader("data/agenticlabs_mission.txt")
# docs = loader.load()

# Simulate loading documents as LangChain Document objects
from langchain_core.documents import Document
docs = [Document(page_content=content) for content in documents_content]

# 1.1. Text Splitting (Chunking)
# We use RecursiveCharacterTextSplitter for more robust chunking in real scenarios.
# For these small docs, it might not split much, but it's good practice.
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=100, # Max characters per chunk
    chunk_overlap=20, # Overlap between chunks to maintain context
    length_function=len,
    is_separator_regex=False,
)
chunks = text_splitter.split_documents(docs)
print(f"Split {len(docs)} documents into {len(chunks)} chunks.")

# 1.2. Embedding Model
# Using a Sentence Transformer model for generating embeddings.
# 'all-MiniLM-L6-v2' is a good balance of performance and speed for demonstration.
embeddings_model = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")
print("Embedding model loaded.")

# 1.3. Vector Storage (ChromaDB)
# Create a Chroma vector store from the document chunks and embeddings.
# This will create an in-memory vector store for this example.
vectorstore = Chroma.from_documents(chunks, embeddings_model)
print("Vector store created and populated with embeddings.")

# Create a retriever object from the vector store
retriever = vectorstore.as_retriever(search_kwargs={"k": 2}) # Retrieve top 2 most relevant chunks
print("--- Indexing Phase Complete ---")

# --- 2. Retrieval Phase: Find Relevant Context ---

print("\n--- Starting Retrieval Phase ---")
query = "What is the main purpose of AgenticLabs.ng and RAG?"
print(f"User Query: '{query}'")

# Retrieve relevant documents based on the query
retrieved_docs = retriever.invoke(query)
print(f"Retrieved {len(retrieved_docs)} relevant documents:")
for i, doc in enumerate(retrieved_docs):
    print(f"  Chunk {i+1}: {doc.page_content}")

print("--- Retrieval Phase Complete ---")

# --- 3. Generation Phase: Formulate the Answer ---

print("\n--- Starting Generation Phase ---")

# Define the prompt template for the LLM
# The 'context' variable will be populated by the retrieved documents.
# The 'question' variable will be the user's original query.
prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI assistant. Use the following context to answer the question. If you don't know the answer, just say that you don't know, don't try to make up an answer.\n\nContext: {context}"),
    ("user", "Question: {question}")
])

# Simulate an LLM call (for a real application, this would be an API call to OpenAI, Google Gemini, Anthropic Claude, etc.)
# We'll use a placeholder function for demonstration.
class MockLLM:
    def invoke(self, prompt_text):
        # In a real scenario, this would send the prompt to an actual LLM and get a response.
        # For this example, we'll simulate a response based on the prompt structure.
        if "AgenticLabs.ng mission" in prompt_text and "RAG systems enhance LLMs" in prompt_text:
            return "AgenticLabs.ng aims to empower developers with AI and automation tools. RAG systems improve LLMs by providing external, up-to-date information, which helps reduce hallucinations."
        elif "don't know" in prompt_text:
            return "I don't have enough information in the provided context to answer that question."
        else:
            return "Based on the provided context, I can answer your question."

mock_llm = MockLLM()

# Construct the RAG chain
# This chain first retrieves documents, then formats them into the prompt, and finally invokes the LLM.
rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt_template
    | mock_llm # Replace with your actual LLM client (e.g., ChatOpenAI, ChatGoogleGenerativeAI)
    | StrOutputParser()
)

# Invoke the RAG chain with the user's query
final_answer = rag_chain.invoke(query)

print(f"\nFinal Answer from LLM: {final_answer}")
print("--- Generation Phase Complete ---")

# Clean up the in-memory vector store (optional, for persistent stores you'd save/load)
vectorstore.delete_collection()
print("\nVector store collection deleted.")


### Interpreting the Code Output and Use Cases

The code above demonstrates a complete, albeit simplified, RAG pipeline. Let's break down what you observed:

1.  **Indexing Phase**: You saw how raw text documents were loaded, split into smaller `chunks`, and then transformed into numerical `embeddings` using the `SentenceTransformerEmbeddings` model. These embeddings, along with their original text, were stored in `ChromaDB`, an in-memory vector store for this example. The `retriever` object was then configured to fetch the top 2 most relevant chunks (`k=2`).
    *   **Output**: You'll see messages confirming the loading, splitting, and vector store creation.

2.  **Retrieval Phase**: When a `query` was provided, the `retriever` took this query, embedded it, and performed a similarity search against all the stored document embeddings in ChromaDB. It successfully identified and returned the chunks most semantically similar to the query.
    *   **Output**: The code prints the user's query and then lists the `page_content` of the retrieved chunks. Notice how these chunks directly contain information relevant to "AgenticLabs.ng mission" and "RAG systems."

3.  **Generation Phase**: This is where the magic happens. The retrieved `context` (the relevant chunks) and the original `question` were combined into a `prompt_template`. This template instructs the LLM to use *only* the provided context to answer the question, significantly reducing the chances of hallucination and ensuring factual accuracy based on your data. The `MockLLM` then simulated an LLM's response, demonstrating how a real LLM would synthesize an answer from the given context.
    *   **Output**: The `Final Answer from LLM` directly reflects the information found in the retrieved chunks, demonstrating how RAG guides the LLM's response.

#### Performance Trade-offs and Considerations:

*   **Indexing Time**: For very large datasets, indexing can be time-consuming and resource-intensive, especially embedding generation. This is typically an offline, batch process.
*   **Embedding Model Choice**: Different embedding models offer varying levels of semantic understanding, speed, and computational cost. Larger models often provide better accuracy but are slower. Smaller models like `all-MiniLM-L6-v2` are excellent for quick demonstrations and many production scenarios.
*   **Vector Database**: The choice of vector database (ChromaDB, Pinecone, Weaviate, Qdrant, FAISS, etc.) impacts scalability, search speed, and cost. In-memory solutions like FAISS or local ChromaDB are great for development, while cloud-hosted solutions are for production.
*   **Retrieval Speed**: The speed of similarity search depends on the vector database, the number of embeddings, and the dimensionality of the embeddings. Optimizations like indexing algorithms (e.g., HNSW) are crucial for large-scale retrieval.
*   **LLM Inference Time**: The final generation step still relies on the LLM's inference speed, which can be a bottleneck. Prompt engineering (like the `prompt_template` used) is vital for guiding the LLM effectively.
*   **Chunking Strategy**: How you split documents (chunk size, overlap) significantly impacts retrieval quality. Too small, and context is lost; too large, and irrelevant information might be retrieved, or the chunk might exceed the embedding model's token limit.

#### Typical Use Cases:

*   **Enterprise Knowledge Bases**: Building internal QA systems that answer questions based on company documents, policies, or technical manuals.
*   **Customer Support Chatbots**: Providing accurate and up-to-date answers to customer queries using product documentation or FAQs.
*   **Research and Development**: Helping researchers quickly find relevant information from vast scientific literature.
*   **Legal and Medical Information Systems**: Ensuring LLM responses are grounded in specific legal precedents or medical guidelines.
*   **Personalized Learning**: Creating adaptive learning systems that retrieve relevant educational content based on a student's progress and questions.

This basic RAG pipeline forms the foundation for much more complex and robust systems. In subsequent lessons, we'll explore advanced techniques for each of these stages to build truly world-class RAG applications.


### Resources

*   **LangChain Documentation**: The primary framework used for orchestrating the RAG pipeline. [https://www.langchain.com/](https://www.langchain.com/)
*   **ChromaDB Documentation**: Learn more about this popular open-source vector database. [https://www.trychroma.com/](https://www.trychroma.com/)
*   **Hugging Face Transformers**: Explore a vast collection of pre-trained models, including embedding models like `sentence-transformers`. [https://huggingface.co/transformers](https://huggingface.co/transformers)
*   **Sentence Transformers Library**: Details on the embedding models used. [https://www.sbert.net/](https://www.sbert.net/)
*   **Google AI Studio / Gemini API**: For integrating powerful generative models into your RAG pipeline. [https://ai.google.dev/](https://ai.google.dev/)
*   **OpenAI API Documentation**: Another leading provider of LLMs for the generation phase. [https://platform.openai.com/docs/](https://platform.openai.com/docs/)
*   **Anthropic Claude**: Explore Anthropic's LLMs for robust generation. [https://www.anthropic.com/api](https://www.anthropic.com/api)
